# Interactive Control System Designer

This notebook provides an interactive dashboard to design PID controllers for linear time-invariant (LTI) systems. By adjusting the proportional ($K_{\mathrm{p}}$), integral ($K_{\mathrm{i}}$), and derivative ($K_{\mathrm{d}}$) gains, you can instantly observe the effects on the system's root locus, step response, and Bode plot.

## 1. Setup and Imports
First, we import the necessary libraries. We rely on the Python `control` systems library for calculations, `matplotlib` for plotting, and `ipywidgets` for the interactive user interface.

In [1]:
import io
import numpy as np
import control as ct
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, FloatText, HBox, VBox, HTML, Output, link
from IPython.display import display, Image

## 2. Dashboard Logic
The function below encapsulates the dashboard logic. It calculates the closed-loop system dynamics, evaluates performance against user-defined limits (overshoot and settling time), and updates the plots responsively.

The following performance metrics are calculated and displayed:
- **Overshoot**: $M_{\mathrm{p}} \le M_{\mathrm{max}} \implies \xi \ge -\dfrac{\log(M_{\mathrm{max}})}{\sqrt{\pi^2 + \log^2(M_{\mathrm{max}})}}$
- **Settling time**: $T_{\mathrm{s}} \approx \dfrac{3}{\xi \omega_{\mathrm{n}}} \le T_{\mathrm{s,max}} \implies \xi \omega_{\mathrm{n}} \ge \dfrac{3}{T_{\mathrm{s,max}}}$
- **Rise time**: $T_{\mathrm{r}} \approx \dfrac{1.8}{\omega_{\mathrm{n}}} \le T_{\mathrm{r,max}} \implies \omega_{\mathrm{n}} \ge \dfrac{1.8}{T_{\mathrm{r,max}}}$

In [2]:
def control_system_designer(num_G, den_G, max_os=5.0, max_ts=30.0):
    """
    Interactive Control System Designer.
    num_G, den_G: Lists of coefficients for the plant transfer function.
    max_os: Maximum allowed overshoot (%).
    max_ts: Maximum allowed settling time (seconds).
    """

    # 1. Define the Plant
    G = ct.tf(num_G, den_G)

    # 2. Calculate Boundaries from Specs
    xi_req = -np.log(max_os/100) / np.sqrt(np.pi**2 + np.log(max_os/100)**2)
    sigma_req = 3 / max_ts  # Using the 3/Ts criterion

    # Create layout widgets for the statistics (HTML) and plots (Output)
    stats_widget = HTML(value="")
    plot_output = Output()

    # 3. The Interactive Dashboard Function
    def update_dashboard(Kp, Ki, Kd):
        if Kp == 0 and Ki == 0 and Kd == 0:
            stats_widget.value = "<b>Please increase at least one gain to start.</b>"
            with plot_output:
                plot_output.clear_output(wait=True)
            return

        # Build the controller dynamically
        s = ct.tf('s')
        C = ct.tf([Kp], [1])
        if Ki != 0:
            C += Ki / s
        if Kd != 0:
            C += Kd * s

        # Calculate Loops
        L = C * G
        T = ct.feedback(L, 1)

        # Calculate metrics and update the stats HTML widget
        poles = T.poles()
        info = ct.step_info(T, SettlingTimeThreshold=0.05)
        os_actual = info.get('Overshoot', 0)
        ts_actual = info.get('SettlingTime', 0)

        os_pass = "\u2705" if os_actual <= max_os else "\u274c"
        ts_pass = "\u2705" if ts_actual <= max_ts else "\u274c"

        stats_widget.value = f"""
        <div style="font-family: monospace; line-height: 1.5; padding: 10px; border-left: 4px solid #ccc; margin-bottom: 15px;">
            <b>--- Measured Performance ---</b><br>
            Overshoot:      {os_actual:.2f}% &nbsp;&nbsp; {os_pass} (Req: &lt;= {max_os}%)<br>
            Settling Time:  {ts_actual:.2f} s &nbsp;&nbsp; {ts_pass} (Req: &lt;= {max_ts} s)<br>
            CL Poles:       {[np.round(p, 3) for p in poles]}
        </div>
        """

        # Render plots inside the Output widget.
        # We use matplotlib.figure.Figure directly (not plt.subplots) to avoid
        # the Jupyter inline backend duplicating the plots. The figure is
        # rendered to a PNG buffer and displayed as an IPython Image.
        with plot_output:
            plot_output.clear_output(wait=True)
            with plt.ioff():
                fig, axs = plt.subplots(1, 3, figsize=(18, 5))

            # --- PLOT 1: Root Locus ---
            ct.root_locus(L, ax=axs[0], grid=False)
            axs[0].plot(poles.real, poles.imag, 'rs', markersize=8, label='Current Poles')

            # Draw boundaries
            axs[0].axvline(-sigma_req, color='g', linestyle='--', label=f'Settling Time (\u03c3 = {-sigma_req:.2f})')
            y_vals = np.linspace(0, max(2, max(abs(poles.imag)) + 1), 50)
            theta = np.arccos(xi_req)
            axs[0].plot(-y_vals/np.tan(theta), y_vals, 'r--', label=f'Overshoot (\u03b6 = {xi_req:.2f})')
            axs[0].plot(-y_vals/np.tan(theta), -y_vals, 'r--')

            axs[0].axhline(0, color='black', lw=1)
            axs[0].axvline(0, color='black', lw=1)
            axs[0].set_title('Root Locus')
            axs[0].legend(loc='lower left', fontsize=8)

            # --- PLOT 2: Step Response ---
            t_sim = max(40, max_ts * 1.5)
            time, response = ct.step_response(T, T=np.linspace(0, t_sim, 1000))

            axs[1].plot(time, response, 'b-', lw=2)
            axs[1].axhline(1, color='k', linestyle='--', alpha=0.7, label='Target (1.0)')

            axs[1].axhline(1 + max_os/100, color='r', linestyle=':', lw=2, label=f'Max OS ({max_os}%)')
            axs[1].axvline(max_ts, color='g', linestyle=':', lw=2, label=f'Max Ts ({max_ts}s)')

            axs[1].set_title('Step Response vs. Specifications')
            axs[1].set_xlabel('Time (s)')
            axs[1].grid(True)
            axs[1].legend(loc='lower right', fontsize=8)

            # --- PLOT 3: Bode Diagram ---
            mag, phase, omega = ct.bode_plot(L, plot=False)
            axs[2].semilogx(omega, 20 * np.log10(mag), 'b-', lw=2)
            axs[2].axhline(0, color='k', linestyle='--', alpha=0.5)
            axs[2].set_title('Open-Loop Bode (Magnitude)')
            axs[2].set_xlabel('Frequency (rad/s)')
            axs[2].set_ylabel('Magnitude (dB)')
            axs[2].grid(True, which='both', ls=':')

            fig.tight_layout()
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
            buf.seek(0)
            display(Image(data=buf.read()))
            plt.close(fig)

    # Create sliders and text inputs for the gains
    slider_Kp = FloatSlider(value=0.0, min=0.0, max=10.0, step=0.01, description='Kp:')
    text_Kp = FloatText(value=0.0, step=0.01, layout={'width': '80px'})
    link((slider_Kp, 'value'), (text_Kp, 'value'))
    box_Kp = HBox([slider_Kp, text_Kp])

    slider_Ki = FloatSlider(value=0.1, min=0.0, max=10.0, step=0.01, description='Ki:')
    text_Ki = FloatText(value=0.1, step=0.01, layout={'width': '80px'})
    link((slider_Ki, 'value'), (text_Ki, 'value'))
    box_Ki = HBox([slider_Ki, text_Ki])

    slider_Kd = FloatSlider(value=0.0, min=0.0, max=5.0, step=0.01, description='Kd:')
    text_Kd = FloatText(value=0.0, step=0.01, layout={'width': '80px'})
    link((slider_Kd, 'value'), (text_Kd, 'value'))
    box_Kd = HBox([slider_Kd, text_Kd])

    controls = VBox([box_Kp, box_Ki, box_Kd])

    # Update function for the observers
    def on_change(change):
        update_dashboard(slider_Kp.value, slider_Ki.value, slider_Kd.value)

    slider_Kp.observe(on_change, names='value')
    slider_Ki.observe(on_change, names='value')
    slider_Kd.observe(on_change, names='value')

    # Trigger initial update
    update_dashboard(slider_Kp.value, slider_Ki.value, slider_Kd.value)

    # Display the structured vertical layout: controls -> statistics -> plots
    display(VBox([controls, stats_widget, plot_output]))

## 3. Define the Plant $G(s)$ and Specifications
Here we define our first test case: a second-order plant.

$$G(s) = \frac{1}{s^2 + 3s + 2}$$

We also establish performance constraints: a maximum overshoot of $5\%$ and a settling time under $30$ seconds.

In [3]:
# Define numerator and denominator for G(s)
num = [1]
den = [1, 3, 2]

# Design goals
max_os = 5.0        # max overshoot (%)
max_ts = 30.0       # max settling time (s)

## 4. Run the Interactive Dashboard
Execute the cell below to spawn the interactive controller tuning interface. Drag the sliders or type a value to see how the system's behavior changes in real-time.

In [4]:
control_system_designer(num, den, max_os, max_ts)

## 5. Additional Test: Higher-Order Plant
Let's test with a more complex, third-order plant to verify the designer's flexibility.

$$G_{\mathrm{test}}(s) = \frac{10}{s^3 + 6s^2 + 11s + 6}$$

We relax the criteria slightly: max overshoot of $10\%$ and max settling time of $15$ seconds.

In [6]:
num_test = [10]
den_test = [1, 6, 11, 6]

control_system_designer(num_test, den_test, max_os=10.0, max_ts=15.0)